In [ ]:
# =========================
# 0) Setup (Colab)
# =========================
!pip -q install rapidfuzz

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.4 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
# =========================
# 1) Imports & Config
# =========================
import os, re, random, datetime, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from rapidfuzz.fuzz import ratio as fuzz_ratio


In [ ]:
# ---------- config ----------
SEED = 42
random.seed(SEED); np.random.seed(SEED)

DATA_DIR = "/content/drive/MyDrive/NEW/Dataset"
FAKE_CSV     = f"{DATA_DIR}/fake.csv"
NONFAKE_CSV  = f"{DATA_DIR}/non-fake.csv"

TEXT_COL, LABEL_COL = "Review", "Label"
FUZZ_THR = 95  # used only for light dedup in aug pool

STAMP   = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = f"{DATA_DIR}/prepared_paper_{STAMP}"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ---------- utils ----------
def norm(s: str) -> str:
    s = str(s).strip()
    return re.sub(r"\s+", " ", s)

def load_labeled_csv(path: str, label_val: int) -> pd.DataFrame:
    df = pd.read_csv(path)
    assert TEXT_COL in df.columns, f"{path} must contain '{TEXT_COL}'"
    df = df[[TEXT_COL]].dropna().copy()
    df[TEXT_COL] = df[TEXT_COL].astype(str).map(norm)
    df[LABEL_COL] = int(label_val)
    return df[(df[TEXT_COL]!="") & df[LABEL_COL].isin([0,1])]

def majority_dedup(df: pd.DataFrame) -> pd.DataFrame:
    """Exact dup w/ conflicts -> keep majority class."""
    from collections import defaultdict
    counts = defaultdict(lambda: [0,0])  # [fake, non-fake]
    for _, r in df.iterrows():
        counts[r[TEXT_COL].lower()][r[LABEL_COL]] += 1
    seen, rows, conflicts = set(), [], 0
    for _, r in df.iterrows():
        k = r[TEXT_COL].lower()
        if k in seen: continue
        seen.add(k)
        c0, c1 = counts[k]
        if c0>0 and c1>0: conflicts += 1
        maj = r[LABEL_COL] if c0==c1 else (0 if c0>c1 else 1)
        rows.append({TEXT_COL: r[TEXT_COL], LABEL_COL: maj})
    out = pd.DataFrame(rows)
    print(f"🧹 Dedup → kept {len(out)} (removed {len(df)-len(out)}), conflicts: {conflicts}")
    return out

def final_dedup(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["__k"] = df[TEXT_COL].str.lower().str.strip()
    return df.drop_duplicates(subset="__k").drop(columns="__k")


In [ ]:
# ----- very-light placeholder augmentation -----
def noise_aug_one(t: str) -> str:
    import random
    # end punctuation tweak
    t = re.sub(r"[!.…]*$", random.choice(["", "!", "!!", "…"]), t)
    # occasional adjacent swap
    if len(t) > 12 and random.random() < 0.30:
        i = random.randint(1, len(t)-2)
        t = t[:i-1] + t[i] + t[i-1] + t[i+1:]
    # occasional thin space
    if len(t) > 20 and random.random() < 0.35:
        j = random.randint(5, min(len(t)-5, 40))
        t = t[:j] + " " + t[j:]
    return norm(t)

def make_augments(base_fake_df: pd.DataFrame, need: int) -> pd.DataFrame:
    if need <= 0 or base_fake_df.empty:
        return pd.DataFrame(columns=[TEXT_COL, LABEL_COL])
    src = base_fake_df[[TEXT_COL]].sample(n=need, replace=True, random_state=SEED)
    rows = [{TEXT_COL: noise_aug_one(s), LABEL_COL: 0} for s in src[TEXT_COL].tolist()]
    aug = pd.DataFrame(rows)
    # small internal dedup of the aug pool
    aug["__k"] = aug[TEXT_COL].str.lower().str.strip()
    aug = aug.drop_duplicates(subset="__k").drop(columns="__k")
    return aug


In [ ]:
# ---------- 1) load originals, clean, global dedup ----------
fake_df    = load_labeled_csv(FAKE_CSV, 0)
nonfake_df = load_labeled_csv(NONFAKE_CSV, 1)
orig_df    = pd.concat([fake_df, nonfake_df], ignore_index=True)
print(f"📥 Originals: total={len(orig_df)} (fake={len(fake_df)}, non-fake={len(nonfake_df)})")

orig_df = majority_dedup(orig_df)

📥 Originals: total=9049 (fake=1339, non-fake=7710)
🧹 Dedup → kept 9033 (removed 16), conflicts: 0


In [ ]:
# ---------- 2) balance first (paper-style): fake == non-fake ----------
target_fake = len(nonfake_df)  # 1:1
need_aug    = max(0, target_fake - len(fake_df))
aug_fake    = make_augments(fake_df, need_aug)

fake_bal = final_dedup(pd.concat([fake_df, aug_fake], ignore_index=True))
if len(fake_bal) > target_fake:
    fake_bal = fake_bal.sample(target_fake, random_state=SEED).reset_index(drop=True)

nonfake_bal = nonfake_df.sample(len(fake_bal), random_state=SEED)
print(f"📦 Balanced pools → fake:{len(fake_bal)} non-fake:{len(nonfake_bal)}")

📦 Balanced pools → fake:6258 non-fake:6258


In [ ]:
# ---------- 3) full balanced corpus + shuffle ----------
full_bal = final_dedup(pd.concat([fake_bal, nonfake_bal], ignore_index=True)).sample(
    frac=1.0, random_state=SEED).reset_index(drop=True)

print("✅ Full balanced size:", len(full_bal),
      "| counts:", full_bal[LABEL_COL].value_counts().to_dict())


✅ Full balanced size: 12506 | counts: {0: 6258, 1: 6248}


In [ ]:
# ---------- 4) stratified 80/10/10 split ----------
trainval, test = train_test_split(
    full_bal, test_size=0.10, stratify=full_bal[LABEL_COL], random_state=SEED
)
train, val = train_test_split(
    trainval, test_size=1/9, stratify=trainval[LABEL_COL], random_state=SEED
)

print(f"✅ Split → Train:{len(train)} Val:{len(val)} Test:{len(test)}")
print("Class counts:",
      {"train": train[LABEL_COL].value_counts().to_dict(),
       "val":   val[LABEL_COL].value_counts().to_dict(),
       "test":  test[LABEL_COL].value_counts().to_dict()})


✅ Split → Train:10004 Val:1251 Test:1251
Class counts: {'train': {0: 5006, 1: 4998}, 'val': {0: 626, 1: 625}, 'test': {0: 626, 1: 625}}


In [ ]:
# ---------- 5) save once, reuse forever ----------
train.to_csv(os.path.join(OUT_DIR, "train.csv"), index=False)
val.to_csv(  os.path.join(OUT_DIR, "val.csv"),   index=False)
test.to_csv( os.path.join(OUT_DIR, "test.csv"),  index=False)

with open(os.path.join(OUT_DIR, "prep_meta.json"), "w", encoding="utf-8") as f:
    json.dump({
        "seed": SEED,
        "mode": "paper_style",
        "paths": {"fake": FAKE_CSV, "nonfake": NONFAKE_CSV},
        "counts": {
            "full_bal": full_bal[LABEL_COL].value_counts().to_dict(),
            "train":    train[LABEL_COL].value_counts().to_dict(),
            "val":      val[LABEL_COL].value_counts().to_dict(),
            "test":     test[LABEL_COL].value_counts().to_dict()
        }
    }, f, ensure_ascii=False, indent=2)

print("\n💾 Saved to:", OUT_DIR)
print("➡️ Use these files in any model runs without re-preprocessing:")
print(os.path.join(OUT_DIR, "train.csv"))
print(os.path.join(OUT_DIR, "val.csv"))
print(os.path.join(OUT_DIR, "test.csv"))


💾 Saved to: /content/drive/MyDrive/NEW/Dataset/prepared_paper_20250915_161745
➡️ Use these files in any model runs without re-preprocessing:
/content/drive/MyDrive/NEW/Dataset/prepared_paper_20250915_161745/train.csv
/content/drive/MyDrive/NEW/Dataset/prepared_paper_20250915_161745/val.csv
/content/drive/MyDrive/NEW/Dataset/prepared_paper_20250915_161745/test.csv
